<a href="https://colab.research.google.com/github/syeda-bisma05/weather-data-analysis/blob/main/weather_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Historical Weather Data Analysis

This notebook explores 11 years (2006 to 2016) of hourly weather data, including temperature, humidity, wind speed, pressure and visibility. It cleans the data, checks for missing values, duplicates and outliers, and uses graphs to find patterns in the weather.
## Questions

**Single column:**
1. How is temperature distributed, and what is the typical value?
2. How is wind speed distributed, and are there extreme values?
3. How often does each precipitation type (rain or snow) occur?

**Two columns:**

4. What is the relationship between temperature and humidity?
5. How does the average temperature change across months?

##Step 1: Basic Analysis

In [1]:
from google.colab import files
uploaded = files.upload()

Saving weather_data.csv to weather_data.csv


In [ ]:
!pip install pandas numpy matplotlib seaborn plotly

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [4]:
df = pd.read_csv('weather_data.csv')

In [5]:
df.shape

(96453, 14)

In [6]:
df.head()

,Formatted Date,Time,Summary,Precip Type,Temperature (C),Apparent Temperature (C),Wind Speed (km/h),Wind Bearing (degrees),Visibility (km),Pressure (millibars),Humidity,Month,Day,Year
0,4/1/2006,00:00:00.000,Partly Cloudy,rain,9.472222,7.388889,14.1197,251.0,15.8263,1015.13,0.89,4,1,2006
1,4/1/2006,01:00:00.000,Partly Cloudy,rain,9.355556,7.227778,14.2646,259.0,15.8263,1015.63,0.86,4,1,2006
2,4/1/2006,02:00:00.000,Mostly Cloudy,rain,9.377778,9.377778,3.9284,204.0,14.9569,1015.94,0.89,4,1,2006
3,4/1/2006,03:00:00.000,Partly Cloudy,rain,8.288889,5.944444,14.1036,269.0,15.8263,1016.41,0.83,4,1,2006
4,4/1/2006,04:00:00.000,Mostly Cloudy,rain,8.755556,6.977778,11.0446,259.0,15.8263,1016.51,0.83,4,1,2006


In [7]:
df.columns

Index(['Formatted Date', 'Time', 'Summary', 'Precip Type', 'Temperature (C)',
       'Apparent Temperature (C)', 'Wind Speed (km/h)',
       'Wind Bearing (degrees)', 'Visibility (km)', 'Pressure (millibars)',
       'Humidity', 'Month', 'Day', 'Year'],
      dtype='object')

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96453 entries, 0 to 96452
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Formatted Date            96453 non-null  object 
 1   Time                      96453 non-null  object 
 2   Summary                   96453 non-null  object 
 3   Precip Type               95936 non-null  object 
 4   Temperature (C)           96453 non-null  float64
 5   Apparent Temperature (C)  96453 non-null  float64
 6   Wind Speed (km/h)         96453 non-null  float64
 7   Wind Bearing (degrees)    96453 non-null  float64
 8   Visibility (km)           96453 non-null  float64
 9   Pressure (millibars)      96453 non-null  float64
 10  Humidity                  96453 non-null  float64
 11  Month                     96453 non-null  int64  
 12  Day                       96453 non-null  int64  
 13  Year                      96453 non-null  int64  
dtypes: flo

In [9]:
df['Formatted Date'] = pd.to_datetime(df['Formatted Date'], format='%m/%d/%Y')
print(df['Formatted Date'].dtype)
print(df['Formatted Date'].min(), df['Formatted Date'].max())

datetime64[ns]
2006-01-01 00:00:00 2016-12-31 00:00:00




**Shape (df.shape):** The dataset has 96,453 rows and 14 columns.

**First rows (df.head()):** Each row is one hourly weather reading. The first rows show the date, time, summary, precipitation type, temperature and other measurements.

**Columns (df.columns):** The data has weather columns (temperature, apparent temperature, wind speed, wind bearing, visibility, pressure, humidity), date columns (Formatted Date, Time, Month, Day, Year) and two text columns (Summary, Precip Type).

**Data types (df.info()):** There are 7 float, 3 integer and 4 object columns. Only Precip Type has missing values (517 out of 96,453). Formatted Date is stored as text (object), so it needs to be converted to a date type.

**Date conversion:** I converted Formatted Date to datetime using pd.to_datetime so I can sort by date, filter by date range and plot trends over time. The data runs from 2006-01-01 to 2016-12-31 (11 years of hourly data).

**Places:** There is no place or country column, so the data covers a single location. There is also no rainfall column, so I use Precip Type (rain or snow) and humidity instead. Because of this, I will compare by Month and Precip Type instead of countries.

**Selecting columns:** I kept only the columns needed for my questions: date, precipitation type, temperature, apparent temperature, wind speed, visibility, pressure, humidity, month and year. I dropped Time, Summary, Wind Bearing and Day because my questions do not use them. The data is now sorted by date.

In [11]:
df.describe()

,Formatted Date,Temperature (C),Apparent Temperature (C),Wind Speed (km/h),Wind Bearing (degrees),Visibility (km),Pressure (millibars),Humidity,Month,Day,Year
count,96453,96453.000000,96453.000000,96453.000000,96453.000000,96453.000000,96453.000000,96453.000000,96453.000000,96453.00000,96453.000000
mean,2011-07-02 09:45:39.386022400,11.932678,10.855029,10.810640,187.509232,10.347325,1003.235956,0.734899,6.523799,15.72708,2011.000404
min,2006-01-01 00:00:00,-21.822222,-27.716667,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.00000,2006.000000
25%,2008-10-01 00:00:00,4.688889,2.311111,5.828200,116.000000,8.339800,1011.900000,0.600000,4.000000,8.00000,2008.000000
50%,2011-07-02 00:00:00,12.000000,12.000000,9.965900,180.000000,10.046400,1016.450000,0.780000,7.000000,16.00000,2011.000000
75%,2014-04-02 00:00:00,18.838889,18.838889,14.135800,290.000000,14.812000,1021.090000,0.890000,10.000000,23.00000,2014.000000
max,2016-12-31 00:00:00,39.905556,39.344444,63.852600,359.000000,16.100000,1046.380000,1.000000,12.000000,31.00000,2016.000000
std,NaN,9.551546,10.696847,6.913571,107.383428,4.192123,116.969906,0.195473,3.448495,8.80216,3.162142


**Summary statistics (df.describe()):** The data has 96,453 hourly records from 2006-01-01 to 2016-12-31.

- **Temperature:** The average is about 11.9 C and the middle value (median) is 12.0 C. It ranges from -21.8 C to 39.9 C, which is realistic for this kind of climate.
- **Wind speed:** The average is about 10.8 km/h, and the maximum is 63.9 km/h. This is far above the 75th percentile (14.1 km/h), so it may be a real storm. I will check it in the outlier step.
- **Humidity:** The average is 0.73 and the median is 0.78. Most values are between 0.60 and 0.89. The minimum is 0, which is suspicious because humidity is almost never exactly zero.
- **Pressure:** The median is about 1016 millibars, but the minimum is 0 and the mean (1003) is lower than the median. Zero pressure is not physically possible, so these values look like data errors. I will check them in the outlier step.
- **Visibility:** The average is about 10.3 km and the maximum is 16.1 km. A minimum of 0 km can happen in heavy fog, so I will check it before treating it as an error.